In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 6 - Week 7
# --------------------------------------------------
# Use all accumulated observations already stored in the Week 7 .npy files.
# Fit an ARD Matern GP, derive search widths from fitted lengthscales,
# generate local + wider + global candidates, then calibrate EI and UCB.

In [2]:
X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy").reshape(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

assert len(X) == len(Y)
assert X.shape[1] == 5

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:", best_x)
print("Current best observed output:", best_y)

X shape: (26, 5)
Y shape: (26,)

Current best observed input: [0.442929 0.409333 0.63183  0.7398   0.129381]
Current best observed output: -0.19517557780237


In [3]:
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.ones(5) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
1.19**2 * Matern(length_scale=[0.704, 0.889, 1.18, 0.814, 0.837], nu=2.5) + WhiteKernel(noise_level=1e-08)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [4]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1 / lengthscales
sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:", lengthscales)
print("Normalised inverse-lengthscale sensitivity:", sensitivity)


ARD lengthscales: [0.70431719 0.8894927  1.17875579 0.81359125 0.83746083]
Normalised inverse-lengthscale sensitivity: [0.2441387  0.19331365 0.14587507 0.21134825 0.20532433]


In [5]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal search widths:", local_scale)
print("Wider search widths:", wide_scale)


Local search widths: [0.1 0.1 0.1 0.1 0.1]
Wider search widths: [0.2 0.2 0.2 0.2 0.2]


In [6]:
rng = np.random.default_rng(42)

local_candidates = best_x + rng.normal(
    0,
    local_scale,
    size=(50000, 5)
)

wide_candidates = best_x + rng.normal(
    0,
    wide_scale,
    size=(20000, 5)
)

global_candidates = rng.uniform(
    0,
    1,
    size=(10000, 5)
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("Generated candidates:", len(candidates))

Generated candidates: 80000


In [7]:
tree = cKDTree(X)

distance, _ = tree.query(candidates, k=1)

candidates = candidates[distance > 0.008]

print("Candidates after filtering:", len(candidates))

Candidates after filtering: 80000


In [8]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

In [9]:
def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        + sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [10]:
print("\nEI calibration:\n")

for xi in [0.0, 0.001, 0.005, 0.01, 0.02]:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        f"xi={xi}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI calibration:

xi=0.0 
 candidate = [0.46388597 0.41659559 0.57639873 0.69305435 0.16822969] 
 mean = -0.187274 
 std = 0.037781 
 EI = 0.01935201 

xi=0.001 
 candidate = [0.46388597 0.41659559 0.57639873 0.69305435 0.16822969] 
 mean = -0.187274 
 std = 0.037781 
 EI = 0.01877435 

xi=0.005 
 candidate = [0.46388597 0.41659559 0.57639873 0.69305435 0.16822969] 
 mean = -0.187274 
 std = 0.037781 
 EI = 0.01656798 

xi=0.01 
 candidate = [0.46388597 0.41659559 0.57639873 0.69305435 0.16822969] 
 mean = -0.187274 
 std = 0.037781 
 EI = 0.01404677 

xi=0.02 
 candidate = [0.45377058 0.38972967 0.55369584 0.68609142 0.19558819] 
 mean = -0.200643 
 std = 0.051923 
 EI = 0.01042345 



In [11]:
print("\nUCB calibration:\n")

for beta in [0.1, 0.25, 0.5, 1.0, 1.5]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB calibration:

beta=0.1 
 candidate = [0.4560372  0.4056988  0.58706228 0.70507511 0.16002345] 
 mean = -0.184029 
 std = 0.030243 
 UCB = -0.181005 

beta=0.25 
 candidate = [0.4560372  0.4056988  0.58706228 0.70507511 0.16002345] 
 mean = -0.184029 
 std = 0.030243 
 UCB = -0.176468 

beta=0.5 
 candidate = [0.46388597 0.41659559 0.57639873 0.69305435 0.16822969] 
 mean = -0.187274 
 std = 0.037781 
 UCB = -0.168383 

beta=1.0 
 candidate = [0.45377058 0.38972967 0.55369584 0.68609142 0.19558819] 
 mean = -0.200643 
 std = 0.051923 
 UCB = -0.14872 

beta=1.5 
 candidate = [0.4559628  0.35822964 0.58325528 0.67176059 0.18206176] 
 mean = -0.208449 
 std = 0.057832 
 UCB = -0.121701 



In [12]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.4560372  0.4056988  0.58706228 0.70507511 0.16002345]
mean = -0.18402914980308105
std = 0.030242674593690164


In [13]:
# --------------------------------------------------
# Final Function 6 Week 7 selection
# --------------------------------------------------
#
# The highest GP predicted mean and low-exploration UCB
# settings (beta=0.1 and beta=0.25) all selected the same
# candidate.
#
# The GP predicts -0.184029 at this point, which is higher
# than the current best observed value of -0.195176.
#
# EI selected a nearby point with slightly greater uncertainty,
# but the low-exploration UCB result gives a stronger predicted
# mean while still retaining an uncertainty component.
#
# I therefore use UCB with beta=0.1 for Week 7.

beta = 0.1

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week7_candidate = candidates[final_idx]

print("Week 7 Function 6 candidate:")
print(week7_candidate)

print("\nPredicted mean:", mu[final_idx])
print("Predicted std:", sigma[final_idx])
print("UCB:", UCB[final_idx])

portal = "-".join(f"{x:.6f}" for x in week7_candidate)

print("\nPortal format:")
print(portal)

Week 7 Function 6 candidate:
[0.4560372  0.4056988  0.58706228 0.70507511 0.16002345]

Predicted mean: -0.18402914980308105
Predicted std: 0.030242674593690164
UCB: -0.18100488234371204

Portal format:
0.456037-0.405699-0.587062-0.705075-0.160023
